In [17]:
import numpy as np
import pandas as pd
import torch

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
df=pd.read_csv('startup_success_dataset.csv', on_bad_lines ='skip')
df

,funding_rounds,founder_experience_years,team_size,market_size_billion,product_traction_users,burn_rate_million,revenue_million,investor_type,sector,founder_background,outcome
0,4,13,58,48.225483,594843,18.519211,1.483962e+06,tier2_vc,Health,academic,IPO
1,1,6,221,31.532647,393020,14.298149,8.620568e+05,tier2_vc,Fintech,first_time,Failure
2,3,5,247,4.969722,27636,20.447567,9.726169e+04,none,SaaS,first_time,Failure
3,3,14,229,3.084209,235376,8.177417,1.145785e+06,none,Ecommerce,ex_bigtech,Acquisition
4,1,17,235,13.818188,391765,4.879792,8.608949e+05,none,Health,first_time,Acquisition
...,...,...,...,...,...,...,...,...,...,...,...
99995,1,10,189,38.766348,261932,7.945616,1.005687e+06,tier1_vc,Ecommerce,serial_founder,Failure
99996,1,15,39,49.983039,355153,93.223490,1.627273e+06,none,AI,first_time,Acquisition
99997,0,22,215,3.086470,606292,12.832589,1.967245e+06,tier1_vc,Health,academic,Failure
99998,4,24,231,10.815435,277717,28.205609,2.757053e+05,angel,Crypto,ex_bigtech,Failure


In [20]:
df.isnull().sum()

,0
funding_rounds,0
founder_experience_years,0
team_size,0
market_size_billion,0
product_traction_users,0
burn_rate_million,0
revenue_million,0
investor_type,0
sector,0
founder_background,0


In [21]:
import pandas as pd

# Convert numerical columns to numeric
numeric_cols = [
    "funding_rounds",
    "founder_experience_years",
    "team_size",
    "market_size_billion",
    "product_traction_users",
    "burn_rate_million",
    "revenue_million"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill numerical missing values using median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values using mode
categorical_cols = [
    "investor_type",
    "sector",
    "founder_background",

]

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Remove rows where target is missing
df = df.dropna(subset=["outcome"])

# Check remaining missing values
print(df.isnull().sum())

funding_rounds              0
founder_experience_years    0
team_size                   0
market_size_billion         0
product_traction_users      0
burn_rate_million           0
revenue_million             0
investor_type               0
sector                      0
founder_background          0
outcome                     0
dtype: int64


In [22]:
X=df.iloc[:,:-1]
Y=df.iloc[:,-1]


In [23]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2)


In [24]:
feature_order = X.columns.tolist()

print(feature_order)

['funding_rounds', 'founder_experience_years', 'team_size', 'market_size_billion', 'product_traction_users', 'burn_rate_million', 'revenue_million', 'investor_type', 'sector', 'founder_background']


In [25]:
import pandas as pd

print(pd.Series(Y_train).value_counts())
print(pd.Series(Y_train).value_counts(normalize=True))

outcome
Failure        44523
Acquisition    33811
IPO             1666
Name: count, dtype: int64
outcome
Failure        0.556538
Acquisition    0.422637
IPO            0.020825
Name: proportion, dtype: float64


In [26]:
from sklearn.preprocessing import LabelEncoder,StandardScaler
le=LabelEncoder()
Y_train=le.fit_transform(Y_train)
Y_test=le.transform(Y_test)
for col in categorical_cols:
    le=LabelEncoder()
    X_train[col]=le.fit_transform(X_train[col])
    X_test[col]=le.transform(X_test[col])
X_train = X_train.dropna(subset=["revenue_million"])

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [28]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self,x,y):
    self.x=torch.tensor(x, dtype=torch.float32)
    self.y=torch.tensor(y, dtype=torch.long)
  def __len__(self):
    return len(self.x)
  def __getitem__(self,idx):
    return self.x[idx],self.y[idx]

In [29]:
train_dataset=CustomDataset(X_train,Y_train)
train_dataLoader= DataLoader(train_dataset, batch_size=128, shuffle= True, pin_memory=True)
test_dataset=CustomDataset(X_test, Y_test)
test_dataLoader= DataLoader(test_dataset, batch_size=128, shuffle= False, pin_memory=True)





In [30]:
import torch
import torch.nn as nn


class mySimpleNN(nn.Module):

    def __init__(self, num_features):

        super().__init__()

        self.model = nn.Sequential(

            # Hidden Layer
            nn.Linear(
                num_features,
                64
            ),

            nn.BatchNorm1d(
                64,
                momentum=0.22224623206758784,
                eps=9.98053794949822e-06
            ),

            nn.ReLU(),

            nn.Dropout(
                0.16985411441075865
            ),

            # Output Layer
            nn.Linear(
                64,
                3
            )
        )

    def forward(self, X):

        return self.model(X)

In [31]:
import torch.optim as optim
import torch.nn as nn

epochs = 100

lr = 0.0007817379480677396

model = mySimpleNN(
    X_train.shape[1]
).to(device)

criterion = (
    nn.CrossEntropyLoss()
)

optimizer = optim.Adam(
    model.parameters(),
    lr=lr
)

In [32]:
model.to(device)

mySimpleNN(
  (model): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=9.98053794949822e-06, momentum=0.22224623206758784, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.16985411441075865, inplace=False)
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)

In [43]:
best_loss = float("inf")
patience = 10
counter = 0

for epoch in range(100):

    model.train()

    epoch_loss = 0

    for batch_features, batch_labels in train_dataLoader:

        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        output = model(batch_features)

        loss = criterion(
            output,
            batch_labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = (
        epoch_loss
        / len(train_dataLoader)
    )

    print(
        f"Epoch [{epoch+1}/100], "
        f"Loss: {avg_loss:.4f}"
    )

    # Early stopping
    if avg_loss < best_loss:

        best_loss = avg_loss
        counter = 0

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

    else:

        counter += 1

    if counter >= patience:

        print(
            f"Stopped early at epoch "
            f"{epoch+1}"
        )

        break

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [1/100], Loss: 0.5462
Epoch [2/100], Loss: 0.5460
Epoch [3/100], Loss: 0.5456
Epoch [4/100], Loss: 0.5462
Epoch [5/100], Loss: 0.5457
Epoch [6/100], Loss: 0.5454
Epoch [7/100], Loss: 0.5459
Epoch [8/100], Loss: 0.5462
Epoch [9/100], Loss: 0.5457
Epoch [10/100], Loss: 0.5453
Epoch [11/100], Loss: 0.5453
Epoch [12/100], Loss: 0.5463
Epoch [13/100], Loss: 0.5464
Epoch [14/100], Loss: 0.5456
Epoch [15/100], Loss: 0.5458
Epoch [16/100], Loss: 0.5468
Epoch [17/100], Loss: 0.5461
Epoch [18/100], Loss: 0.5452
Epoch [19/100], Loss: 0.5455
Epoch [20/100], Loss: 0.5452
Epoch [21/100], Loss: 0.5454
Epoch [22/100], Loss: 0.5457
Epoch [23/100], Loss: 0.5455
Epoch [24/100], Loss: 0.5456
Epoch [25/100], Loss: 0.5463
Epoch [26/100], Loss: 0.5460
Epoch [27/100], Loss: 0.5455
Epoch [28/100], Loss: 0.5454
Stopped early at epoch 28


In [44]:
total=0
correct=0
with torch.no_grad():
  for batch_features, batch_labels in test_dataLoader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    out=model(batch_features)
    _,predicted=torch.max(out,1)
    total=total+batch_labels.shape[0]
    correct=correct+(predicted==batch_labels).sum().item()
print(f"Accuracy: {correct/total}")


Accuracy: 0.74395


In [46]:
import pandas as pd
import torch

new_data_dict = {
    'funding_rounds': 4,
    'founder_experience_years': 8,
    'team_size': 120,
    'market_size_billion': 15,
    'product_traction_users': 800000,
    'burn_rate_million': 4,
    'revenue_million': 30,
    'investor_type': 1,
    'sector': 2,
    'founder_background': 1
}
# maintain correct feature order
new_data = [[
    new_data_dict[col]
    for col in feature_order
]]

# convert to DataFrame for scaler
new_df = pd.DataFrame(
    new_data,
    columns=feature_order
)

# scale input
new_scaled = scaler.transform(new_df)

# tensor
new_tensor = torch.tensor(
    new_scaled,
    dtype=torch.float32
).to(device)

model.eval()

with torch.no_grad():

    output = model(new_tensor)

    # predicted class
    prediction = torch.argmax(
        output,
        dim=1
    )
outcomeList=df["outcome"].unique()
print("Raw output:", output.cpu().numpy())
print(outcomeList)
print("Prediction:", outcomeList[prediction.item()])

Raw output: [[ 0.90736574 -0.04674494 -4.094352  ]]
['IPO' 'Failure' 'Acquisition']
Prediction: IPO


In [47]:
model.eval()

with torch.no_grad():

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32
    ).to(device)

    Y_train_tensor = torch.tensor(
        Y_train,
        dtype=torch.long
    ).to(device)

    output = model(X_train_tensor)

    # predicted class index
    _, prediction = torch.max(
        output,
        dim=1
    )

    accuracy_train = (
        prediction == Y_train_tensor
    ).float().mean()

print(
    f"Accuracy for training data: "
    f"{accuracy_train.item()*100:.2f}%"
)

Accuracy for training data: 74.38%


In [48]:
model.eval()

with torch.no_grad():

    X_test_tensor = torch.tensor(
        X_test,
        dtype=torch.float32
    ).to(device)

    Y_test_tensor = torch.tensor(
        Y_test,
        dtype=torch.long
    ).to(device)

    output = model(X_test_tensor)

    # predicted class
    _, prediction = torch.max(
        output,
        dim=1
    )

    accuracy_test = (
        prediction == Y_test_tensor
    ).float().mean()

print(
    f"Accuracy for testing data: "
    f"{accuracy_test.item()*100:.2f}%"
)

Accuracy for testing data: 74.30%


In [55]:
torch.save(
    model.state_dict(),
    "startup_model.pth"
)

In [68]:
torch.save(
    model.state_dict(),
    "startup_prediction_model.pth"
)

In [69]:
%cd /content

/content


In [39]:
import torch
import torch.nn as nn


class MLPNet(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim,
        num_hidden_layers,
        neurons_per_layer,
        dropout_rate=0.3,
        bn_momentum=0.1,
        bn_eps=1e-5
    ):

        super().__init__()

        layers = []

        in_features = input_dim

        for _ in range(num_hidden_layers):

            layers.append(
                nn.Linear(
                    in_features,
                    neurons_per_layer
                )
            )

            # BatchNorm only if enabled

            layers.append(
                nn.BatchNorm1d(
                    neurons_per_layer,
                    momentum=bn_momentum,
                    eps=bn_eps
                )
            )

            layers.append(
                nn.ReLU()
            )

            layers.append(
                nn.Dropout(
                    dropout_rate
                )
            )

            in_features = (
                neurons_per_layer
            )

        layers.append(
            nn.Linear(
                in_features,
                output_dim
            )
        )

        self.net = nn.Sequential(
            *layers
        )

    def forward(self, X):

        return self.net(X)

In [40]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.4 MB/s eta 0:00:00


In [41]:
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


def objective(trial):

    # -----------------------------
    # Hyperparameters
    # -----------------------------

    num_hidden_layers = trial.suggest_int(
        "num_hidden_layers",
        1,
        4
    )

    neurons_per_layer = (
        trial.suggest_categorical(
            "neurons_per_layer",
            [32, 64, 128, 256]
        )
    )

    learning_rate = (
        trial.suggest_float(
            "learning_rate",
            1e-5,
            1e-2,
            log=True
        )
    )

    batch_size = (
        trial.suggest_categorical(
            "batch_size",
            [32, 64, 128]
        )
    )

    optimizer_name = (
        trial.suggest_categorical(
            "optimizer",
            ["Adam", "AdamW", "RMSprop"]
        )
    )

    dropout_rate = (
        trial.suggest_float(
            "dropout_rate",
            0.1,
            0.5
        )

    )
    bn_momentum = (
    trial.suggest_float(
        "bn_momentum",
        0.01,
        0.3
    )
)

    bn_eps = (
      trial.suggest_float(
          "bn_eps",
          1e-6,
          1e-3,
          log=True
      )
  )

    # -----------------------------
    # Model
    # -----------------------------

    model = MLPNet(
        input_dim=X_train.shape[1],
        output_dim=3,
        num_hidden_layers=num_hidden_layers,
        neurons_per_layer=neurons_per_layer,
        dropout_rate=dropout_rate
    ).to(device)

    # -----------------------------
    # Optimizer
    # -----------------------------

    optimizer = getattr(
        torch.optim,
        optimizer_name
    )(
        model.parameters(),
        lr=learning_rate
    )

    # multiclass loss
    criterion = (
        nn.CrossEntropyLoss()
    )

    # -----------------------------
    # DataLoader
    # -----------------------------

    train_dataset = (
        CustomDataset(
            X_train,
            Y_train
        )
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    # -----------------------------
    # Training
    # -----------------------------

    epochs=20

    for epoch in range(epochs):

        model.train()

        for (
            batch_features,
            batch_labels
        ) in train_loader:

            batch_features = (
                batch_features.to(
                    device
                )
            )

            batch_labels = (
                batch_labels.to(
                    device
                )
            )

            outputs = model(
                batch_features
            )

            loss = criterion(
                outputs,
                batch_labels
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

    # -----------------------------
    # Validation
    # -----------------------------

    model.eval()

    with torch.no_grad():

        X_test_tensor = (
            torch.tensor(
                X_test,
                dtype=torch.float32
            )
            .to(device)
        )

        Y_test_tensor = (
            torch.tensor(
                Y_test,
                dtype=torch.long
            )
            .to(device)
        )

        outputs = model(
            X_test_tensor
        )

        _, preds = torch.max(
            outputs,
            dim=1
        )

        accuracy = (
            (
                preds
                == Y_test_tensor
            )
            .float()
            .mean()
            .item()
        )

    return accuracy

In [42]:
import optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

[I 2026-05-25 11:19:02,663] A new study created in memory with name: no-name-2abe770b-943f-4530-8236-2a9da1818dd3
[I 2026-05-25 11:21:22,175] Trial 0 finished with value: 0.7443000078201294 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 32, 'learning_rate': 0.002229564404688017, 'batch_size': 32, 'optimizer': 'RMSprop', 'dropout_rate': 0.29853488857763877, 'bn_momentum': 0.0573069228419336, 'bn_eps': 1.1939924806340676e-06}. Best is trial 0 with value: 0.7443000078201294.
[W 2026-05-25 11:22:02,111] Trial 1 failed with parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 64, 'learning_rate': 4.3327382040191976e-05, 'batch_size': 64, 'optimizer': 'RMSprop', 'dropout_rate': 0.40923461978062525, 'bn_momentum': 0.1046358345920017, 'bn_eps': 0.00029899744180919227} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values =

KeyboardInterrupt: 

In [ ]:
study.best_params